In [ ]:
import copy
from IPython.display import display
import logging
import numpy
import os
import pandas
import pdb
import plotly
import pprint
import pyarrow
import pyarrow.parquet as pq
import six
import sys

plotly.offline.init_notebook_mode(connected=True)

In [ ]:
LOG = logging.getLogger(__name__)
LOG.setLevel(logging.DEBUG)

# create a file handler
handler = logging.StreamHandler()
handler.setLevel(logging.DEBUG)

# create a logging format
formatter = logging.Formatter('%(asctime)s | %(levelname)s | %(message)s', '%H-%M-%S')
handler.setFormatter(formatter)

# add the handlers to the logger
LOG.addHandler(handler)

In [ ]:
DATA_DIR = '2014_Targa_Sixty-Six'
STRIDES = [1, 10]

In [ ]:
FILE_NAMES = [x for x in os.listdir(DATA_DIR) if (x != '250lm.parquet' and x.endswith('.parquet'))]
FILE_NAMES.sort()
LOG.debug(FILE_NAMES)

In [ ]:
COLUMNS_ORIG = ['time', 'handwheelAngle', 'throttle', 'brake', 'altitude', 'horizontalSpeed', 'vxCG', 'vyCG', 'yawAngle', 'pitchAngle', 'rollAngle', 'distance']  # 'latitude', 'longitude'

COLUMNS_TO_DIFF = ['yawAngle', 'pitchAngle', 'rollAngle', 'horizontalSpeed', 'distance', 'vxCG', 'vyCG']
COLUMN_DIFF_PREFIX = 'diff_'
COLUMNS = copy.deepcopy(COLUMNS_ORIG)
for column in COLUMNS_TO_DIFF:
    new_column = COLUMN_DIFF_PREFIX + column
    COLUMNS.append(new_column)

COLUMNS_WITH_GPS_JUMP = ['horizontalSpeed', 'vxCG', 'vyCG', 'yawAngle', 'pitchAngle', 'rollAngle']
DIFF_COLUMNS_WITH_GPS_JUMP = [COLUMN_DIFF_PREFIX + x for x in COLUMNS_WITH_GPS_JUMP]

# load data from parquet files

In [ ]:
DATA = {}

for file_name in FILE_NAMES:
    file_path = os.path.join(DATA_DIR, file_name)
    
    # read parquet
    table = pq.read_table(file_path)
    
    # convert parquet table to pandas dataframe
    df = table.to_pandas()
    
    # save
    DATA[file_name] = {
        'orig': df
    }
    #sig_names = sorted(list(df.columns), key=lambda s: s.lower())
    
    LOG.debug('loaded %s', file_path)

# data size

In [ ]:
data_size_total = 0

for file_name, dfs in six.iteritems(DATA):
    df = dfs['orig']
    LOG.debug('%s : %s' % (file_name, len(df)))
    data_size_total += len(df)

LOG.debug('total: %s', data_size_total)

# data size of interesting samples (either throttle or brake != 0)

In [ ]:
data_size_reduced = 0

for file_name, dfs in six.iteritems(DATA):
    df = dfs['orig']
    df_reduced = df.loc[(df['throttle'] != 0) | (df['brake'] != 0)]
    
    LOG.debug('%s : %s (%s)', file_name, len(df_reduced), 100.0 * len(df_reduced) / len(df))
    data_size_reduced += len(df_reduced)

LOG.debug('total: %s', data_size_reduced)

# add columns for discrete derivatives

- need to correct diff_yawAngle for when wheel axis flips from positive to negative

In [ ]:
#fix_count = 0
#spot_check_freq = 1000

for file_name, dfs in six.iteritems(DATA):
    for stride in STRIDES:
        LOG.debug('%s stride: %s', file_name, stride)
        
        df = dfs['orig'].copy(deep=True)
        for column in COLUMNS_TO_DIFF:
            new_column = COLUMN_DIFF_PREFIX + column
            
            df[new_column] = df[column] - df[column].shift(stride)
        
        
        # correct for yawAngle sign flip
        indexes = df.index[(df['diff_yawAngle'] > 300) | (df['diff_yawAngle'] < -300)]
        for i in indexes:
            #fix_count += 1
            #if fix_count % spot_check_freq == 0:
            #    display(df.iloc[i-2:i+2][['yawAngle', 'diff_yawAngle']])

            #old_value = df.iloc[i]['diff_yawAngle']
            df.at[i, 'diff_yawAngle'] = (180 - abs(df.iloc[i]['yawAngle'])) + (180 - abs(df.iloc[i-stride]['yawAngle']))
            
            #if fix_count % spot_check_freq == 0:
            #    display(df.iloc[i-1:i+2][['yawAngle', 'diff_yawAngle']])
            #    LOG.warning('%s:%s fixed diff_yawAngle at %s (%s -> %s)', file_name, stride, i, round(old_value, 4), round(df.iloc[i]['diff_yawAngle'], 4))
        LOG.warning('# diff_yawAngle fixed: %s' % len(indexes))
        
        # replace NaN with zeros in diff columns
        values = {COLUMN_DIFF_PREFIX+x:0 for x in COLUMNS_TO_DIFF}
        df.fillna(value=values, inplace=True)
        
        DATA[file_name][stride] = df
    
    continue

### print max discrete derivative per column

In [ ]:
for file_name, dfs in six.iteritems(DATA):
    for stride in STRIDES:
        LOG.debug('%s stride: %s', file_name, stride)
        df = dfs[stride]
        for column in COLUMNS_TO_DIFF:
            diff_column = COLUMN_DIFF_PREFIX + column
            LOG.debug(' - %s: %s', diff_column, max(abs(df[diff_column])))

# clean discontinuities

In [ ]:
thresholds =[
    ('diff_yawAngle', 10),
    ('diff_pitchAngle', 2),
    ('diff_rollAngle', 2),
    ('diff_distance', 10),
    ('diff_vxCG', 10),
    ('diff_vyCG', 10)
]

for file_name, dfs in six.iteritems(DATA):
    LOG.info(file_name)
    for stride in STRIDES:
        LOG.info('stride: %s', stride)
        df = dfs[stride]
        
        for column, threshold in thresholds:
            indexes = df.index[(df[column] > threshold) | (df[column] < -threshold)]
            for i in indexes:
                if numpy.isnan(df.iloc[i-stride]['altitude']):
                    LOG.warning('GPS jump at %s : %s -> %s', i, 
                                df.iloc[i][DIFF_COLUMNS_WITH_GPS_JUMP].to_string(header=False, index=False).replace(os.linesep, ', '), 
                                df.iloc[i-1][DIFF_COLUMNS_WITH_GPS_JUMP].to_string(header=False, index=False).replace(os.linesep, ', '))
                    df.at[i, DIFF_COLUMNS_WITH_GPS_JUMP] = df.iloc[i-stride][DIFF_COLUMNS_WITH_GPS_JUMP]

### inspect discontinuities

In [ ]:
manually_inspect = False

thresholds = [
    ('diff_yawAngle', 10),
    ('diff_pitchAngle', 2),
    ('diff_rollAngle', 2),
    ('diff_distance', 10),
    ('diff_vxCG', 10),
    ('diff_vyCG', 10)
]
discontinuities = {
    'count': {
        'total': 0
    },
    'indexes': {}
}

for file_name, dfs in six.iteritems(DATA):
    LOG.info(file_name)
    discontinuities['count'][file_name] = {}
    discontinuities['indexes'][file_name] = {}
    
    for stride in STRIDES:
        LOG.info('stride: %s', stride)
        df = dfs[stride]
        discontinuities['count'][file_name][stride] = 0
        discontinuities['indexes'][file_name][stride] = []
        
        for column, threshold in thresholds:
            indexes = df.index[(df[column] > threshold) | (df[column] < -threshold)]
            indexes_list = indexes.to_list()
            
            if manually_inspect:
                for i in indexes:
                    LOG.debug('%s : %s : %s', column, i, df.iloc[i][column])
                    display(df.iloc[i-5:i+5,:][COLUMNS])

                    pdb.set_trace()
            
            discontinuities['count']['total'] += len(indexes_list)
            discontinuities['count'][file_name][stride] += len(indexes_list)
            discontinuities['indexes'][file_name][stride].extend(indexes_list)

pprint.pprint(discontinuities['count'])

# view data tables

In [ ]:
for file_name, dfs in six.iteritems(DATA):
    df = dfs['orig']
    print('%s : %s' % (file_name, len(df)))
    display(df[COLUMNS].head())
    display(df[COLUMNS].tail())

# plot data

In [ ]:
num_points = 5000
stride = 10

data = {}
i = 0

for file_name, df in six.iteritems(DATA):
    data[file_name] = []
    time = df['time'].values

    for sig_name in COLUMNS_TO_DIFF:
        trace = plotly.graph_objs.Scattergl(
            x = time[:num_points:stride],
            y = df[sig_name].values[:num_points:stride],
            name = sig_name,
            yaxis = 'y'
        )
        
        if max(abs(trace['y'])) > 1:
            trace['yaxis'] = 'y2'
            trace['name'] = sig_name + ' (right)'
        
        data[file_name].append(trace)
    
    layout = plotly.graph_objs.Layout(
        title=file_name,
        yaxis=dict(
            side='left'
        ),
        yaxis2=dict(
            overlaying='y',
            side='right'
        )
    )
    
    fig = plotly.graph_objs.Figure(data=data[file_name], layout=layout)
    plotly.offline.iplot(fig)